## Noise configuration

In [1]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    pauli_error,
    ReadoutError,
)
import numpy as np
import pennylane as qml
from pennylane_qiskit import AerDevice          # pip install pennylane-qiskit

NOISE_RATES = {
    "p_d":    1.5e-4,   # single-qubit gate depolarizing
    "p_dep":  8e-4,     # state-prep / measurement depolarizing
    "p_d1":   7.5e-4,   # two-qubit gate depolarizing (qubit 1)
    "p_d2":   7.5e-4,   # two-qubit gate depolarizing (qubit 2)
    "p_alpha": 1e-4,    # imprecise single-qubit rotation  R_V(p_α)
    "p_xx":   1e-3,     # two-qubit imprecise rotation     H(p_xx)
    "p_h":    1.25e-3,  # ion heating                      H(p_h)
}

# Gates present in the hardware-compiled L-VQE ansatz
_SINGLE_QUBIT_GATES = ["rx", "ry", "rz", "u1", "u2", "u3", "h", "s", "sdg", "t", "tdg", "x", "y", "z"]
_TWO_QUBIT_GATES    = ["cx", "cz", "ecr", "rzz", "rxx"]


def build_paper_noise_model(rates: dict = NOISE_RATES) -> NoiseModel:
    """
    Constructs a Qiskit Aer NoiseModel that matches the noise channels
    described in Eqs. (19)–(22) of the paper.

    Channel mapping
    ---------------
    W(p_dep)   →  depolarizing_error on reset/measure  +  state-prep wrap
    R_V(p_α)   →  equal-weight Pauli {X,Y,Z} error after every 1q rotation
    H(p_xx)    →  (X⊗X) Pauli error after every 2q gate  [p_xx]
    H(p_h)     →  (X⊗X) Pauli error after every 2q gate  [p_h]  (ion heating)
    W(p_d)     →  depolarizing_error on every 1q gate
    W(p_d1/2)  →  depolarizing_error on every 2q gate
    """
    nm = NoiseModel()
    p = rates

    # ── 1. Single-qubit gate errors ───────────────────────────────────────────
    # W(p_d): depolarizing after every 1q gate
    dep_1q = depolarizing_error(p["p_d"], 1)

    # R_V(p_α): imprecise rotation — uniform Pauli {X,Y,Z} flip (Eq. 20)
    # Each of X, Y, Z has probability p_α/3  so total flip prob = p_α
    r_alpha = pauli_error(
        [("X", p["p_alpha"] / 3),
         ("Y", p["p_alpha"] / 3),
         ("Z", p["p_alpha"] / 3),
         ("I", 1 - p["p_alpha"])],
    )
    combined_1q = dep_1q.compose(r_alpha)
    nm.add_all_qubit_quantum_error(combined_1q, _SINGLE_QUBIT_GATES)

    # ── 2. Two-qubit gate errors ──────────────────────────────────────────────
    # W(p_d1) ⊗ W(p_d2): depolarizing on each qubit of the pair
    dep_2q = depolarizing_error(p["p_d1"], 2)          # Qiskit 2q depolarizing

    # H(p_xx): imprecise XX rotation (Eq. 21)
    h_xx = pauli_error(
        [("XX", p["p_xx"]),
         ("II", 1 - p["p_xx"])],
    )

    # H(p_h): ion heating — same channel structure, different rate
    h_heat = pauli_error(
        [("XX", p["p_h"]),
         ("II", 1 - p["p_h"])]    )

    combined_2q = dep_2q.compose(h_xx).compose(h_heat)
    nm.add_all_qubit_quantum_error(combined_2q, _TWO_QUBIT_GATES)

    # ── 3. Measurement error  W(p_dep) on POVM (Eq. 19 + text) ──────────────
    # "measurement error is modeled by preceding the ideal POVM element
    #  with an action of the depolarizing channel"
    # For a single qubit, depolarizing with p_dep shrinks the Bloch vector.
    # We approximate this as a symmetric bit-flip readout error:
    #   P(0|1) = P(1|0) ≈ p_dep * 2/3   (matches first-order Bloch shrinkage)
    p_ro = p["p_dep"] * 2 / 3
    ro_error = ReadoutError([[1 - p_ro, p_ro],
                             [p_ro,     1 - p_ro]])
    nm.add_all_qubit_readout_error(ro_error)

    # ── 4. State-preparation error  W(p_dep)ρ₀ ───────────────────────────────
    # Applied as a depolarizing channel on the |0⟩ reset operation
    prep_error = depolarizing_error(p["p_dep"], 1)
    nm.add_all_qubit_quantum_error(prep_error, ["reset"])

    return nm


def make_noisy_aer_device(n_q: int, shots: int, rates: dict = NOISE_RATES):
    """
    Returns a PennyLane device backed by AerSimulator with the paper noise model.
    Requires: pennylane-qiskit  (pip install pennylane-qiskit)
    """
    noise_model = build_paper_noise_model(rates)
    backend = AerSimulator(noise_model=noise_model)

    # pennylane-qiskit exposes the Aer backend directly
    dev = qml.device(
        "qiskit.aer",
        wires=n_q,
        shots=shots,
        backend=backend,
    )
    return dev

In [2]:
import csv
import datetime
import numpy as np
import networkx as nx
from l_vqe_engine import (
    best_known_maxcut_cost,
    build_maxcut_hamiltonian,
    simulate_one_vqe_with_device,
    simulate_one_lvqe_with_device
)
SHOTS = 1000
N_SEEDS = 10

G = nx.random_regular_graph(d=3, n=8,  seed=42)
rng_noisy = np.random.default_rng(42)

n_nodes = G.number_of_nodes()
n_qubits = G.number_of_nodes()   # 1 qubit per node for MaxCut
true_baseline = best_known_maxcut_cost(G)
H_maxcut = build_maxcut_hamiltonian(G)

vqe_modularities = []
vqe_rhos         = []
vqe_costs        = []

for trial in range(N_SEEDS):
    print(f"  VQE trial {trial+1}/{N_SEEDS} ...", end=" ")
    
    # Fresh noisy device per trial (ensures independent noise samples)
    trial_dev = make_noisy_aer_device(n_qubits, shots=SHOTS)
    
    rng_trial = np.random.default_rng(trial)   # different seed each trial
    
    result = simulate_one_vqe_with_device(
        n_q=n_qubits,
        H=H_maxcut,
        n_layers=2,
        shots=SHOTS,
        max_evals=500,
        rng=rng_trial,
        optimizer="COBYLA",
        dev=trial_dev,
    )
    
    mod  = -result["final_cost"]
    rho  = mod / true_baseline
    vqe_modularities.append(mod)
    vqe_rhos.append(rho)
    vqe_costs.append(result["final_cost"])
    print(f"ρ = {rho:.4f}")

print(f"\nVQE  ρ:  {np.mean(vqe_rhos):.4f} ± {np.std(vqe_rhos):.4f}")

  VQE trial 1/10 ... 

c:\Users\myria\Desktop\Delft\QIST\q3\aqa_delft\layered_vqe\.venv\Lib\site-packages\pennylane\devices\legacy_facade.py:212: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


  Final VQE Cost: -9.326000
ρ = -0.9326
  VQE trial 2/10 ...   Final VQE Cost: -9.518000
ρ = -0.9518
  VQE trial 3/10 ...   Final VQE Cost: -8.976000
ρ = -0.8976
  VQE trial 4/10 ...   Final VQE Cost: -9.397000
ρ = -0.9397
  VQE trial 5/10 ...   Final VQE Cost: -9.245000
ρ = -0.9245
  VQE trial 6/10 ...   Final VQE Cost: -9.601000
ρ = -0.9601
  VQE trial 7/10 ...   Final VQE Cost: -9.002000
ρ = -0.9002
  VQE trial 8/10 ...   Final VQE Cost: -9.569000
ρ = -0.9569
  VQE trial 9/10 ...   Final VQE Cost: -9.060000
ρ = -0.9060
  VQE trial 10/10 ...   Final VQE Cost: -9.570000
ρ = -0.9570

VQE  ρ:  -0.9326 ± 0.0233


In [3]:
lvqe_modularities = []
lvqe_rhos         = []
lvqe_costs        = []
for trial in range(N_SEEDS):
    print(f"  LVQE trial {trial+1}/{N_SEEDS} ...", end=" ")
    
    # Fresh noisy device per trial (ensures independent noise samples)
    trial_dev = make_noisy_aer_device(n_qubits, shots=SHOTS)
    
    rng_trial = np.random.default_rng(trial)   # different seed each trial
    
    lvqe_results = simulate_one_lvqe_with_device(
    n_q=n_qubits,
    H=H_maxcut,
    max_layers=2,
    shots=SHOTS,
    max_iter_per_layer=100,
    rng=rng_trial,
    optimizer="COBYLA",
    dev=trial_dev,
)
    
    mod  = -lvqe_results["final_cost"]
    rho  = mod / true_baseline
    lvqe_modularities.append(mod)
    lvqe_rhos.append(rho)
    lvqe_costs.append(lvqe_results["final_cost"])
    print(f"ρ = {rho:.4f}")

print(f"\nLVQE  ρ:  {np.mean(lvqe_rhos):.4f} ± {np.std(lvqe_rhos):.4f}")

  LVQE trial 1/10 ...   Layer 0  (8 params) ... cost = -9.543000
  Layer 1  (36 params) ... cost = -9.889000
  Layer 2  (64 params) ... cost = -9.905000
ρ = -0.9863
  LVQE trial 2/10 ...   Layer 0  (8 params) ... cost = -7.872000
  Layer 1  (36 params) ... cost = -9.757000
  Layer 2  (64 params) ... cost = -9.776000
ρ = -0.9713
  LVQE trial 3/10 ...   Layer 0  (8 params) ... cost = -8.040000
  Layer 1  (36 params) ... cost = -9.232000
  Layer 2  (64 params) ... cost = -9.686000
ρ = -0.9625
  LVQE trial 4/10 ...   Layer 0  (8 params) ... cost = -9.976000
  Layer 1  (36 params) ... cost = -9.993000
  Layer 2  (64 params) ... cost = -9.964000
ρ = -0.9936
  LVQE trial 5/10 ...   Layer 0  (8 params) ... cost = -9.297000
  Layer 1  (36 params) ... cost = -9.739000
  Layer 2  (64 params) ... cost = -9.750000
ρ = -0.9691
  LVQE trial 6/10 ...   Layer 0  (8 params) ... cost = -9.986000
  Layer 1  (36 params) ... cost = -9.991000
  Layer 2  (64 params) ... cost = -9.973000
ρ = -0.9967
  LVQE tri

In [8]:

import os


print(f"\nLVQE  ρ:  {np.mean(lvqe_rhos):.4f} ± {np.std(lvqe_rhos):.4f}")

# ── Updated comparison table (mean ± std across trials) ──────────────────────
print("\n" + "="*58)
print(f"{'Metric':<28} {'L-VQE':>14}  {'VQE':>14}")
print("="*58)
print(f"{'Mean modularity':<28} {np.mean(lvqe_modularities):>14.5f}  {np.mean(vqe_modularities):>14.5f}")
print(f"{'Std modularity':<28} {np.std(lvqe_modularities):>14.5f}  {np.std(vqe_modularities):>14.5f}")
print(f"{'Mean approx ratio ρ':<28} {np.mean(lvqe_rhos):>14.4f}  {np.mean(vqe_rhos):>14.4f}")
print(f"{'Std approx ratio ρ':<28} {np.std(lvqe_rhos):>14.4f}  {np.std(vqe_rhos):>14.4f}")
print(f"{'True baseline':<28} {true_baseline:>14.5f}  {true_baseline:>14.5f}")
print(f"{'Trials':<28} {N_SEEDS:>14}  {N_SEEDS:>14}")
print("="*58)

# ── Save to CSV ───────────────────────────────────────────────────────────────
SUMMARY_CSV = "vqe_vs_lvqe_noisy_maxcut.csv"

rows = []
for trial in range(N_SEEDS):
    rows.append({
        "trial":         trial,
        "method":        "L-VQE",
        "final_cost":    lvqe_costs[trial],
        "modularity":    lvqe_modularities[trial],
        "approx_ratio":  lvqe_rhos[trial],
        "shots":         SHOTS,
        "true_baseline": true_baseline,
    })
    rows.append({
        "trial":         trial,
        "method":        "VQE",
        "final_cost":    vqe_costs[trial],
        "modularity":    vqe_modularities[trial],
        "approx_ratio":  vqe_rhos[trial],
        "shots":         SHOTS,
        "true_baseline": true_baseline,
    })

write_header = not os.path.exists(SUMMARY_CSV)
with open(SUMMARY_CSV, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    if write_header:
        writer.writeheader()
    writer.writerows(rows)

print(f"\nPer-trial results saved → '{SUMMARY_CSV}'")



LVQE  ρ:  -0.9781 ± 0.0120

Metric                                L-VQE             VQE
Mean modularity                     9.78090         9.32640
Std modularity                      0.12021         0.23257
Mean approx ratio ρ                 -0.9781         -0.9326
Std approx ratio ρ                   0.0120          0.0233
True baseline                     -10.00000       -10.00000
Trials                                   10              10

Per-trial results saved → 'vqe_vs_lvqe_noisy_maxcut.csv'


In [5]:
# from l_vqe_engine import simulate_one_vqe_with_device


# vqe_results = simulate_one_vqe_with_device(
#     n_q=n_qubits,
#     H=H_maxcut,
#     n_layers=2,          # same depth as final L-VQE circuit
#     shots=SHOTS,
#     max_evals=500,   # same total budget
#     rng=rng_noisy,
#     optimizer="COBYLA",
#     dev=trial_dev,
# )
 
# vqe_modularity = -vqe_results["final_cost"]
# vqe_rho        = vqe_modularity / true_baseline
# print(f"  Modularity:        {vqe_modularity:.6f}")
# print(f"  Approximation ρ:   {vqe_rho:.4f}")
 


In [7]:
# SUMMARY_CSV = "vqe_vs_lvqe_noisy_max_cut.csv"
# MAX_ITER = 100
# MAX_LAYERS = 2
# vqe_total_budget = 500

# shared_meta = {
#     "graph":          "regular_graph(n=8, d=3)",
#     "n_nodes":        n_nodes,
#     "n_qubits":       n_qubits,
#     "max_layers":     MAX_LAYERS,
#     "shots":          SHOTS,
#     "optimizer":      "COBYLA",
#     "true_baseline":  true_baseline,
#     # noise rates
#     "p_d":            NOISE_RATES["p_d"],
#     "p_dep":          NOISE_RATES["p_dep"],
#     "p_d1":           NOISE_RATES["p_d1"],
#     "p_d2":           NOISE_RATES["p_d2"],
#     "p_alpha":        NOISE_RATES["p_alpha"],
#     "p_xx":           NOISE_RATES["p_xx"],
#     "p_h":            NOISE_RATES["p_h"],
# }
 
# rows = [
#     {**shared_meta,
#      "method":          "L-VQE",
#      "total_budget":    MAX_ITER * MAX_LAYERS + MAX_ITER * 3,
#      "final_cost":      lvqe_results["final_cost"],
#      "modularity":      lvqe_modularity,
#      "approx_ratio":    lvqe_rho},
#     {**shared_meta,
#      "method":          "VQE",
#      "total_budget":    vqe_total_budget,
#      "final_cost":      vqe_results["final_cost"],
#      "modularity":      vqe_modularity,
#      "approx_ratio":    vqe_rho},
# ]
 
# write_header = True
# try:
#     with open(SUMMARY_CSV, "r"):
#         write_header = False
# except FileNotFoundError:
#     pass
 
# with open(SUMMARY_CSV, "a", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=rows[0].keys())
#     if write_header:
#         writer.writeheader()
#     writer.writerows(rows)
 
# print(f"\nSummary saved → '{SUMMARY_CSV}'")
 
# # ── Save cost histories ───────────────────────────────────────────────────────
# HISTORY_CSV = "vqe_vs_lvqe_noisy_cost_history.csv"
 
# with open(HISTORY_CSV, "w", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=["method", "step", "cost"])
#     writer.writeheader()
#     for i, cost in enumerate(lvqe_results["cost_history"]):
#         writer.writerow({"method": "L-VQE", "step": i, "cost": cost})
#     for i, cost in enumerate(vqe_results["cost_history"]):
#         writer.writerow({"method": "VQE",   "step": i, "cost": cost})
 
# print(f"Cost histories saved → '{HISTORY_CSV}'")
 
# # ── Console comparison table ──────────────────────────────────────────────────
# print("\n" + "="*46)
# print(f"{'Metric':<28} {'L-VQE':>8}  {'VQE':>8}")
# print("="*46)
# print(f"{'Final cost':<28} {lvqe_results['final_cost']:>8.5f}  {vqe_results['final_cost']:>8.5f}")
# print(f"{'Modularity':<28} {lvqe_modularity:>8.5f}  {vqe_modularity:>8.5f}")
# print(f"{'Approx ratio ρ':<28} {lvqe_rho:>8.4f}  {vqe_rho:>8.4f}")
# print(f"{'Evaluations':<28} {len(lvqe_results['cost_history']):>8}  {len(vqe_results['cost_history']):>8}")
# print("="*46)

In [ ]:
print(true_baseline)

-10.0
